In [3]:
import nltk

def ngrams(sentence, n):
    words = sentence.split()
    ngrams = zip(*[words[i:] for i in range(n)])
    return list(ngrams)

sentence = '안녕하세요. 만나서 진심으로 반가워요.'

unigram = ngrams(sentence, 1)
bigram = ngrams(sentence, 2)
trigram = ngrams(sentence, 3)

print(unigram)
print(bigram)
print(trigram)

unigram = nltk.ngrams(sentence.split(), 1)
bigram = nltk.ngrams(sentence.split(), 2)
trigram = nltk.ngrams(sentence.split(), 3)

print(list(unigram))
print(list(bigram))
print(list(trigram))


[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요.')]
[('안녕하세요.',), ('만나서',), ('진심으로',), ('반가워요.',)]
[('안녕하세요.', '만나서'), ('만나서', '진심으로'), ('진심으로', '반가워요.')]
[('안녕하세요.', '만나서', '진심으로'), ('만나서', '진심으로', '반가워요.')]


# 벡터화

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    'That movie is famous movie',
    'I like that actor',
    "I don't like that actor"
]

tfidf_vectorizer = TfidfVectorizer()
tfidf_vectorizer.fit(corpus)
tfidf_matrix = tfidf_vectorizer.transform(corpus)

print(tfidf_matrix.toarray())
print(tfidf_vectorizer.vocabulary_)

[[0.         0.         0.39687454 0.39687454 0.         0.79374908
  0.2344005 ]
 [0.61980538 0.         0.         0.         0.61980538 0.
  0.48133417]
 [0.4804584  0.63174505 0.         0.         0.4804584  0.
  0.37311881]]
{'that': 6, 'movie': 5, 'is': 3, 'famous': 2, 'like': 4, 'actor': 0, 'don': 1}


In [ ]:
# 맥용
# import os
# os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

In [5]:
import torch.nn as nn

class VanillaSkipgram(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = embedding_dim
        )
        self.linear = nn.Linear(
            in_features = embedding_dim,
            out_features = vocab_size
        )
    def forward(self, input_ids):
        embeddings = self.embedding(input_ids)
        output = self.linear(embeddings)
        return output

In [6]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load('nsmc')
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 56.0MB/s]                            
[nsmc] download ratings_test.txt: 4.90MB [00:00, 33.3MB/s]                            


In [7]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])

[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


In [8]:
from collections import Counter

def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab

vocab = build_vocab(corpus = tokens, n_vocab = 5000, special_tokens = ['<unk>'])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['<unk>', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


In [9]:
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)
            center_word = sentence[idx]
            context_words  = sentence[window_start:idx] + sentence[idx + 1 : window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs

word_pairs = get_word_pairs(tokens, window_size = 2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


In [ ]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id['<unk>']
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs

index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))   

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
5001


In [11]:
import torch
from torch.utils.data import TensorDataset, DataLoader

index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:,0]
context_indexes = index_pairs[:,1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

In [18]:
import torch.optim as optim

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

word2vec = VanillaSkipgram(vocab_size = len(token_to_id), embedding_dim = 128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr = 0.1)

cuda


In [19]:
import torch.optim as optim

for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss
    cost = cost / len(dataloader)
    print(epoch + 1, cost)

1 tensor(6.1955, device='cuda:0', grad_fn=<DivBackward0>)
2 tensor(5.9813, device='cuda:0', grad_fn=<DivBackward0>)
3 tensor(5.9318, device='cuda:0', grad_fn=<DivBackward0>)
4 tensor(5.9016, device='cuda:0', grad_fn=<DivBackward0>)
5 tensor(5.8796, device='cuda:0', grad_fn=<DivBackward0>)
6 tensor(5.8617, device='cuda:0', grad_fn=<DivBackward0>)
7 tensor(5.8472, device='cuda:0', grad_fn=<DivBackward0>)
8 tensor(5.8341, device='cuda:0', grad_fn=<DivBackward0>)
9 tensor(5.8227, device='cuda:0', grad_fn=<DivBackward0>)
10 tensor(5.8122, device='cuda:0', grad_fn=<DivBackward0>)


In [20]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding
index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)


연기
tensor([-0.9941,  0.0095,  0.0589,  0.1554, -0.1830,  0.1026, -0.4612,  0.3336,
         1.7943,  0.0266, -2.2352, -1.7360,  1.0058, -0.1402,  1.2386,  0.6967,
        -0.6222,  0.1576,  0.9649,  0.7528, -0.4915, -0.4065, -0.5284,  1.2206,
        -0.2786,  0.5071,  0.6861,  0.8035,  0.7851,  1.2102,  1.7735,  0.1594,
         0.9961,  1.2637, -0.6497, -1.4338,  0.1182, -1.2316, -0.4225, -0.7056,
        -0.7469,  0.9510,  0.1631, -0.2984, -1.1968,  0.9631,  0.4010,  1.1554,
        -1.2045, -1.3581,  1.0688,  1.4749,  0.4299, -1.0110, -0.2202,  1.3776,
         0.1184,  0.9016, -0.9320, -1.0388, -0.4192, -1.1208, -0.2331,  1.5564,
         0.5561, -1.6713,  0.3126, -0.1352,  1.0113,  0.5396,  0.6276,  0.6802,
        -0.8147, -0.9416,  1.0303, -0.3327,  0.6646, -0.7323, -0.1899,  0.2425,
        -1.0612,  1.2933, -0.2672,  1.3138,  0.1233, -1.5729, -0.1063,  1.2022,
         0.7059,  0.5985,  0.3713,  1.9596,  1.6593, -0.0940, -0.5376,  0.9491,
         0.4123,  0.7187, -0.8053, -0

In [21]:
import numpy as np
from numpy.linalg import norm

def cosine_similarity(a, b):
    cosine = np.dot(b, a) / (norm(b, axis =1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1: n + 1]
    return top_n

cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n = 5)

for index in top_n:
    print(id_to_token[index], cosine_matrix[index])

연기력 0.33664376
한테 0.29981428
황금 0.27102485
트랜스포머 0.2693696
전도연 0.26578182


# 26. 9. 16